### インストール手順について
公式ドキュメントのとおり、lime_jp は pip でインストールできます。日本語トークナイザー（Sudachi）を使う場合は extras を指定してインストールしてください。

- 最新版をGitからインストール（日本語オプションを含む）:
```
pip install 'git+https://github.com/noyuri2z/lime_jp.git#egg=lime[jp]'
```
- ローカルソースを開発モードでインストール:
```
pip install -e '/Users/noyuritsuji/lime_jp[jp]'
```

ノートブック環境では、以下のセルにある Python 経由の pip コマンドでも同様にセットアップできます。

## LIME_JP チュートリアル

LIME_JPはRibeiro et al. (2021)によって開発されたPythonモジュールLIMEの日本語バージョンになります。主に自然言語処理系のタスクを行うモデルの出力をプロットに加えて日本語の文章でも説明する機能を兼ね備えています。（画像処理、表計算などのモデルにつきましてはまだ未対応となります。ご了承ください。）なお文書分類タスクやテキストマイニングのタスクであれば、それぞれの言葉に対する重みの計算結果が保存できればどのモデルを使っていただいても構いません。

今回の例ではlivedoorニュースが公開している実際のニュースとフェイクニュースの両方を含んだ日本語データセットを使用して簡単な文書分類タスクを行います。その後LIME_JPのexplanier機能を利用して日本語の自然言語処理の結果を出力します。

In [1]:
# lime_jpをインストール
%pip install -e '/Users/noyuritsuji/lime_jp[jp]'

Obtaining file:///Users/noyuritsuji/lime_jp
  Installing build dependencies ... -done
  Checking if build backend supports build_editable ... one
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Getting requirements to build editable ... -done
  Preparing editable metadata (pyproject.toml) ... one
  Preparing editable metadata (pyproject.toml) ... -done
done
  Building editable for lime (pyproject.toml) ... Building wheels for collected packages: lime
  Building editable for lime (pyproject.toml) ... -done
  Created wheel for lime: filename=lime-0.2.0.1-0.editable-py3-none-any.whl size=3877 sha256=217163a789ffbd08f89a28cfced06f2271c62853114a72673e573a5f08d4ecda
  Stored in directory: /private/var/folders/ym/wmpd_g1d5197r4pt0cbdfykm0000gn/T/pip-ephem-wheel-cache-w0e9nciz/wheels/6c/97/6e/181711dd9226e06b737df64c0c74d8ca6d7d45c0eedacb11c3
Successfully built lime
  Attempting uninstall: lime
    Found existing installation: l

In [2]:
# 必要パッケージをインストール（公式ドキュメント準拠のpipインストール）
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
    'stopwordsiso', 'pandas', 'numpy', 'scikit-learn', 'sudachipy', 'sudachidict_core', 'lime'
])

0

### Tf-idfを使った分類モデル

まずはsklearnの回帰モデルを使って簡単な分類モデルを構築します。句読点や助詞などをstopwordのリストを使って省いてからモデルに流します。TokenizerはSudachiPy以外のものを使っても問題ないですが、LIME内では便宜上SudachiPyを使っているため、元のモデルを使用する際にも統一することをお勧めします。

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from stopwordsiso import stopwords
import re
import lime

from sudachipy import dictionary, tokenizer
from lime.lime_text import LimeTextExplainer

# データの読み込み
df = pd.read_csv('fakenews.csv')
df = df.dropna(subset=['context', 'isfake'])

# 日本語のストップワードの読み込み
japanese_stopwords = set(stopwords("ja"))

# フェイクニュースの場合は1, 実際のニュースの場合は0とラベリング
df['label'] = (df['isfake'] > 0).astype(int)

tokenizer_obj = dictionary.Dictionary().create()
mode = tokenizer.Tokenizer.SplitMode.C

# 句読点を取り除くためのリスト
PUNCT_PATTERN = re.compile(r"^[\s\u3000。、！？「」『』（）［］【】.,!?()\"'`~:;<>/\[\]{}|+=\-—–…]+$")

def sudachi_tokenizer(text):
    tokens = []
    for m in tokenizer_obj.tokenize(text, mode):
        surface = m.surface()

        # 句読点のトークンは排除する
        if PUNCT_PATTERN.match(surface):
            continue

        # ストップワードも排除する
        if surface in japanese_stopwords:
            continue

        tokens.append(surface)

    return tokens

X = df['context']
y = df['label']

# 学習データとテストデータに分ける
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# モデルを実行
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        tokenizer=sudachi_tokenizer,
        token_pattern=None,
        max_features=20000
    )),
    ("clf", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

# 正確性のスコアを出力
pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
print(classification_report(y_test, pred))


/opt/homebrew/lib/python3.11/site-packages/stopwordsiso/_core.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Accuracy: 0.8528
              precision    recall  f1-score   support

           0       0.87      0.59      0.70       766
           1       0.85      0.96      0.90      1842

    accuracy                           0.85      2608
   macro avg       0.86      0.78      0.80      2608
weighted avg       0.85      0.85      0.84      2608



元のモデルのF1スコアは0.831となっています。

---
ここからはフェイクニュースと分類された記事の中からサンプルを取り、フェイクニュースと判定された経緯を探っていきます。

In [4]:
print("LIMEで説明文を作成中...")

# フェイクニュースと判定された記事の中から例を選ぶ
fake_indices = np.where(model.predict(X_test) == 1)[0]
idx = fake_indices[1]
text_instance = X_test.iloc[idx]

# LIMEのTextExplainerで説明文とプロットを出力する
explainer = LimeTextExplainer(
    class_names=['Real', 'Fake'],
    split_expression=sudachi_tokenizer
)

exp = explainer.explain_instance(
    text_instance,
    model.predict_proba,
    num_features=10,
    num_samples=500,
    labels=[0, 1]
)

# ノートブック上に表示
# exp.show_in_notebook(text=True)

LIMEで説明文を作成中...


In [5]:
# 元のモデルが計算したフェイクである確率を抽出する
proba_original = model.predict_proba([text_instance])[0, 1]
true_label = "Fake" if y_test.iloc[idx] == 1 else "Real"

print("元のモデルが計算したフェイクである確率 =", round(proba_original, 4))
print("実際のラベル:", true_label)

元のモデルが計算したフェイクである確率 = 0.8811
実際のラベル: Fake


In [6]:
from lime.lime_text import LimeTextExplainer, summarize_lime_explanation_jp
jp_sentences = summarize_lime_explanation_jp(exp, class_idx=1)
print("\n".join(jp_sentences))

このインスタンスは0.881対0.119でFakeと分類されました。Fakeへの分類に最も強い影響を与えた言葉は年, 2020, 出版で、それぞれの重み=+0.037（弱）, 重み=+0.021（弱）, 重み=+0.015（弱）となっています。
他にFakeへの分類の確率を上げた言葉として編集者 (重み=+0.009（弱）)、部 (重み=-0.035（弱）)、インターネット (重み=-0.020（弱）)などが挙げられます。Realへの分類への確率を上げた言葉として、部 (重み=+0.035（弱）)、インターネット (重み=+0.020（弱）)、長 (重み=+0.010（弱）)などが挙げられます。


この結果を見るとなぜフェイク判定に影響した言葉についてのパターンはあまり浮かんでこないですが、

### Random Forestモデルを利用したモデル

In [7]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

from stopwordsiso import stopwords
from sudachipy import dictionary, tokenizer
from lime.lime_text import LimeTextExplainer

# ニュースのデータの読み込み
df = pd.read_csv("fakenews.csv")
df = df.dropna(subset=["context", "isfake"])

# フェイクニュースを1、実際のニュースを0と分類
df["label"] = (df["isfake"] > 0).astype(int)

X = df["context"]
y = df["label"]

# 学習データとテストデータに分ける
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ストップワードの設定
japanese_stopwords = set(stopwords("ja"))

tokenizer_obj = dictionary.Dictionary().create()
mode = tokenizer.Tokenizer.SplitMode.C

# 句読点を除くためのリスト
PUNCT_PATTERN = re.compile(
    r"^[\s\u3000。、！？「」『』（）［］【】.,!?()\"'`~:;<>/\[\]{}|+=\-—–…]+$"
)

# 句読点やストップワード以外の言葉をトークン化する
def sudachi_tokenizer(text):
    tokens = []
    for m in tokenizer_obj.tokenize(text, mode):
        surface = m.surface()

        if PUNCT_PATTERN.match(surface):
            continue
        if surface in japanese_stopwords:
            continue

        tokens.append(surface)

    return tokens

# ランダムフォレストモデルを実装
rf_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        tokenizer=sudachi_tokenizer,
        token_pattern=None,
        max_features=20000
    )),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

# F1スコアなどを出力
pred = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred):.4f}")
print(classification_report(y_test, pred))


Accuracy: 0.8179
              precision    recall  f1-score   support

           0       0.88      0.44      0.59       766
           1       0.81      0.97      0.88      1842

    accuracy                           0.82      2608
   macro avg       0.84      0.71      0.74      2608
weighted avg       0.83      0.82      0.80      2608



In [8]:
# 元のモデルが計算したフェイクである確率を抽出する
proba_original = model.predict_proba([text_instance])[0, 1]
true_label = "Fake" if y_test.iloc[idx] == 1 else "Real"

print("元のモデルが計算したフェイクである確率 =", round(proba_original, 4))
print("実際のラベル:", true_label)

元のモデルが計算したフェイクである確率 = 0.8811
実際のラベル: Fake


In [9]:
print("LIMEで説明文を作成中...")

# フェイクニュースと判定された記事の中から例を選ぶ
fake_indices = np.where(model.predict(X_test) == 1)[0]
idx = fake_indices[1]
text_instance = X_test.iloc[idx]

# LIMEのTextExplainerで説明文とプロットを出力する
explainer = LimeTextExplainer(
    class_names=['Real', 'Fake'],
    split_expression=sudachi_tokenizer
)

exp = explainer.explain_instance(
    text_instance,
    model.predict_proba,
    num_features=10,
    num_samples=500,
    labels=[0, 1]
)

# ノートブック上に表示
# exp.show_in_notebook(text=True)

LIMEで説明文を作成中...


In [10]:
# 元のモデルが計算したフェイクである確率を抽出する
proba_original = model.predict_proba([text_instance])[0, 1]
true_label = "Fake" if y_test.iloc[idx] == 1 else "Real"

print("元のモデルが計算したフェイクである確率 =", round(proba_original, 4))
print("実際のラベル:", true_label)

元のモデルが計算したフェイクである確率 = 0.8811
実際のラベル: Fake


In [11]:
from lime.lime_text import LimeTextExplainer, summarize_lime_explanation_jp
jp_sentences = summarize_lime_explanation_jp(exp, class_idx=1)
print("\n".join(jp_sentences))

このインスタンスは0.881対0.119でFakeと分類されました。Fakeへの分類に最も強い影響を与えた言葉は年, 2020, 出版で、それぞれの重み=+0.037（弱）, 重み=+0.019（弱）, 重み=+0.013（弱）となっています。
他にFakeへの分類の確率を上げた言葉としてof (重み=+0.011（弱）)、部 (重み=-0.037（弱）)、インターネット (重み=-0.018（弱）)などが挙げられます。Realへの分類への確率を上げた言葉として、部 (重み=+0.037（弱）)、インターネット (重み=+0.018（弱）)、長 (重み=+0.011（弱）)などが挙げられます。


In [12]:
print("LIMEで説明文を作成中...")

# フェイクニュースと判定された記事の中から例を選ぶ
fake_indices = np.where(model.predict(X_test) == 1)[0]
idx = fake_indices[2]
text_instance = X_test.iloc[idx]

# LIMEのTextExplainerで説明文とプロットを出力する
explainer = LimeTextExplainer(
    class_names=['Real', 'Fake'],
    split_expression=sudachi_tokenizer
)

exp = explainer.explain_instance(
    text_instance,
    model.predict_proba,
    num_features=10,
    num_samples=500,
    labels=[0, 1]
)

# ノートブック上に表示
#exp.show_in_notebook(text=True)

LIMEで説明文を作成中...


In [13]:
# 元のモデルが計算したフェイクである確率を抽出する
proba_original = model.predict_proba([text_instance])[0, 1]
true_label = "Fake" if y_test.iloc[idx] == 1 else "Real"

print("元のモデルが計算したフェイクである確率 =", round(proba_original, 4))
print("実際のラベル:", true_label)

元のモデルが計算したフェイクである確率 = 0.7027
実際のラベル: Fake


In [14]:
jp_sentences = summarize_lime_explanation_jp(exp)
print("\n".join(jp_sentences))

このインスタンスは0.703対0.297でFakeと分類されました。Fakeへの分類に最も強い影響を与えた言葉は年, 言っ, よるで、それぞれの重み=+0.168（強）, 重み=+0.044（弱）, 重み=-0.105（強）となっています。
他にFakeへの分類の確率を上げた言葉としてスポーツ (重み=-0.051（中）)、病院 (重み=-0.043（弱）)、わかっ (重み=-0.037（弱）)などが挙げられます。Realへの分類への確率を上げた言葉として、よる (重み=+0.105（強）)、スポーツ (重み=+0.051（中）)、病院 (重み=+0.043（弱）)などが挙げられます。


実際の記事と判定されたインスタンスについても見てみましょう。

In [15]:
print("LIMEで説明文を作成中...")

# 実際の記事と判定された記事の中から例を選ぶ
real_indices = np.where(model.predict(X_test) == 0)[0]
idx = real_indices[2]
text_instance = X_test.iloc[idx]

# LIMEのTextExplainerで説明文とプロットを出力する
explainer = LimeTextExplainer(
    class_names=['Real', 'Fake'],
    split_expression=sudachi_tokenizer
)

exp = explainer.explain_instance(
    text_instance,
    model.predict_proba,
    num_features=20,
    num_samples=500,
    labels=[0, 1]
)

# ノートブック上に表示
# exp.show_in_notebook(text=True)

LIMEで説明文を作成中...


In [16]:
# 元のモデルが計算したフェイクである確率を抽出する
proba_original = model.predict_proba([text_instance])[0, 1]
true_label = "Fake" if y_test.iloc[idx] == 1 else "Real"

print("元のモデルが計算したフェイクである確率 =", round(proba_original, 4))
print("実際のラベル:", true_label)

元のモデルが計算したフェイクである確率 = 0.1268
実際のラベル: Real


In [17]:
jp_sentences = summarize_lime_explanation_jp(exp)
print("\n".join(jp_sentences))

このインスタンスは0.873対0.127でRealと分類されました。Realへの分類に最も強い影響を与えた言葉はさん, よる, 9で、それぞれの重み=+0.128（強）, 重み=+0.104（強）, 重み=+0.077（中）となっています。
他にRealへの分類の確率を上げた言葉として歳 (重み=+0.059（中）)、毎日 (重み=+0.051（中）)、コメント (重み=+0.050（中）)などが挙げられます。Fakeへの分類への確率を上げた言葉として、山本 (重み=+0.128（強）)、番組 (重み=+0.057（中）)、衆議院議員 (重み=+0.032（弱）)などが挙げられます。


## 結果の分析

これらのケースから考察できるパターンとして、同じような漢字が3-4回以上出現するとその漢字からフェイク記事ではないかという判定がなされていることがわかります。また固有名詞（インターネットなど）や英単語（Yahoo, ofなど）は本物の記事の特徴として捉えられていることもわかります。スパム記事などではよく同じような言葉が短い文の中に何度も出現するためこの判定方法はある意味的を得ていますが、記事の中でキーパーソンとなっている人物の名前などもフェイクの疑惑対象として捉えられているため、簡易的なデータを増やしたりモデルを改善する必要があると考えられます。